# Parcial 2 - Análisis de datos del mercado laboral GEIH junio 2026

In [0]:
import pandas as pd
from scipy import stats

In [0]:
#Leyendo las tablas de los módulos de la GEIH
ft = spark.read.format("delta").table("workspace.electiva.ft")
desempleados = spark.read.format("delta").table("workspace.electiva.desempleados")
cgse = spark.read.format("delta").table("workspace.electiva.cgse")
ocupados = spark.read.format("delta").table("workspace.electiva.ocupados")

# Convertir el DataFrame de Spark a Pandas
ft = ft.toPandas()
desempleados = desempleados.toPandas()
cgse = cgse.toPandas()
ocupados = ocupados.toPandas()

In [0]:
#Primero hay que anexar la P3271 (sexo) y la P6040 (edad) a las tablas de fuerza de trabajo, desempleados y ocupados
ft = pd.merge(ft, cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271','P6040']],
              on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='left')

desempleados = pd.merge(desempleados, cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271','P6040']],
                        on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='left')

ocupados = pd.merge(ocupados, cgse[['DIRECTORIO', 'SECUENCIA_P', 'ORDEN', 'P3271','P6040']],
                    on=['DIRECTORIO', 'SECUENCIA_P', 'ORDEN'], how='left')

In [0]:
# Aseguramos que la edad y el ingreso sean números
ft['P6040'] = pd.to_numeric(ft['P6040'], errors='coerce')
desempleados['P6040'] = pd.to_numeric(desempleados['P6040'], errors='coerce')
ocupados['P6040'] = pd.to_numeric(ocupados['P6040'], errors='coerce')
ocupados['INGLABO'] = pd.to_numeric(ocupados['INGLABO'], errors='coerce')

In [0]:
# Diccionarios de mapeo
dpto_nombre = {
    5: 'ANTIOQUIA', 8: 'ATLANTICO', 11: 'BOGOTA', 13: 'BOLIVAR', 15: 'BOYACA',
    17: 'CALDAS', 18: 'CAQUETA', 19: 'CAUCA', 20: 'CESAR', 23: 'CORDOBA',
    25: 'CUNDINAMARCA', 27: 'CHOCO', 41: 'HUILA', 44: 'GUAJIRA', 47: 'MAGDALENA',
    50: 'META', 52: 'NARIÑO', 54: 'NORTE DE SANTANDER', 63: 'QUINDIO', 66: 'RISARALDA',
    68: 'SANTANDER', 70: 'SUCRE', 73: 'TOLIMA', 76: 'VALLE DEL CAUCA', 81: 'ARAUCA',
    85: 'CASANARE', 86: 'PUTUMAYO', 88: 'SAN ANDRES', 91: 'AMAZONAS', 94: 'GUAINIA',
    95: 'GUAVIARE', 97: 'VAUPES', 99: 'VICHADA'
}

sexo_nombre = {1:'Hombre', 2:'Mujer'}

In [0]:
# Población total
PT = cgse['FEX_C18'].sum()

# Población en edad de trabajar (PET)
PET = ft[ft['PET']==1]['FEX_C18'].sum()

# Población que no está en edad de trabajar
NO_PET = PT - PET

# Población económicamente activa (PEA)
PEA = ft[ft['FT']==1]['FEX_C18'].sum()

# Población económicamente inactiva (PEI)
PEI = PET - PEA

# Desempleados
D = desempleados[desempleados['DSI']==1]['FEX_C18'].sum()

# Ocupados
O = PEA - D

print(f"población total: {PT}")
print(f"población en edad de trabajar: {PET}")
print(f"población que no está en edad de trabajar: {NO_PET}")
print(f"población económicamente activa: {PEA}")
print(f"población económicamente inactiva: {PEI}")
print(f"desempleados: {D}")
print(f"ocupados: {O}")

In [0]:
# Tasa de ocupación = O/PET*100
TO = O/PET*100
print(f'La tasa de ocupación en Colombia para junio 2026 es de {TO} %')

# Tasa de desempleo = D/PEA*100
TD = D/PEA*100
print(f'La tasa de desempleo en Colombia para junio 2026 es de {TD} %')

**Respuesta:**

A junio de 2026 en Colombia hay cerca de 41 millones de personas en edad de trabajar y unos 11 millones que todavía no tienen la edad y de los que pueden trabajar, unos 26,6 millones trabajan o buscan trabajo, y unos 14,6 millones no lo hacen porque estudian, se dedican al hogar o por otras razones. La tasa de ocupación es de 59,4%, es decir que de cada 100 personas en edad de trabajar, unas 59 tienen empleo. La tasa de desempleo es de 8%, o sea que de cada 100 personas que quieren trabajar, 8 no consiguen empleo

## 2. Tasas de desempleo para hombres y mujeres en todos los departamentos ¿cuáles son los departamentos con mayores y menores tasas de desempleo?

In [0]:
# Desempleados por departamento y sexo
desempleados_dpto_genero = desempleados[desempleados['DSI']==1].groupby(['P3271','DPTO'])['FEX_C18'].sum()

# Población económicamente activa por departamento y sexo
pea_dpto_genero = ft[ft['FT']==1].groupby(['P3271','DPTO'])['FEX_C18'].sum()

tabla_desempleo = pd.DataFrame({'PEA': pea_dpto_genero, 'DESEMPLEADOS': desempleados_dpto_genero})
tabla_desempleo['TD'] = (tabla_desempleo['DESEMPLEADOS']/tabla_desempleo['PEA'])*100

tabla_desempleo = tabla_desempleo.reset_index()
tabla_desempleo['Departamento'] = tabla_desempleo['DPTO'].map(dpto_nombre)
tabla_desempleo['Sexo'] = tabla_desempleo['P3271'].map(sexo_nombre)

tabla_desempleo = tabla_desempleo[['Departamento','Sexo','PEA','DESEMPLEADOS','TD']]
tabla_desempleo

In [0]:
# Hombres ordenados de mayor a menor tasa de desempleo
hombres = tabla_desempleo[tabla_desempleo['Sexo']=='Hombre'].sort_values(by='TD', ascending=False)
hombres

In [0]:
# Mujeres ordenadas de mayor a menor tasa de desempleo
mujeres = tabla_desempleo[tabla_desempleo['Sexo']=='Mujer'].sort_values(by='TD', ascending=False)
mujeres

In [0]:
# Los 3 departamentos con mayor y con menor tasa para cada sexo
print("Hombres - mayores")
print(hombres.head(3)[['Departamento','TD']])
print("Hombres - menores")
print(hombres.tail(3)[['Departamento','TD']])
print("Mujeres - mayores")
print(mujeres.head(3)[['Departamento','TD']])
print("Mujeres - menores")
print(mujeres.tail(3)[['Departamento','TD']])

**Respuesta:**

En los hombres, el desempleo es más alto en San Andrés, Putumayo y Arauca, y más bajo en Vaupés, Cauca y Risaralda y en las mujeres, es más alto en Putumayo, Sucre y Vichada, y más bajo en Vaupés, Amazonas y Huila por ejemplo en Putumayo aparece entre los más altos en los dos casos y Vaupés entre los más bajos también en los dos, ya en general las mujeres tienen más desempleo que los hombres, porque pasa en 28 de los 33 departamentos

## 3. Para cada departamento ¿cuál es el grupo de edad con mayor y menor tasa de desempleo? de 18 a 24, 24 a 30, 30 a 40, 40 a 50 y 60 en adelante

In [0]:
# Creamos la variable grupo de edad. Las personas de 50 a 59 años no entran en el análisis
rangos = [18, 24, 30, 40, 50, 60, 130]
nombres_grupos = ['18 a 24', '24 a 30', '30 a 40', '40 a 50', '50 a 60', '60 en adelante']

ft['grupo_edad'] = pd.cut(ft['P6040'], bins=rangos, labels=nombres_grupos, right=False)
desempleados['grupo_edad'] = pd.cut(desempleados['P6040'], bins=rangos, labels=nombres_grupos, right=False)

In [0]:
# Desempleados y PEA por departamento y grupo de edad
desempleados_dpto_edad = desempleados[desempleados['DSI']==1].groupby(['DPTO','grupo_edad'], observed=True)['FEX_C18'].sum()
pea_dpto_edad = ft[ft['FT']==1].groupby(['DPTO','grupo_edad'], observed=True)['FEX_C18'].sum()

tabla_edad = pd.DataFrame({'PEA': pea_dpto_edad, 'DESEMPLEADOS': desempleados_dpto_edad})
tabla_edad['DESEMPLEADOS'] = tabla_edad['DESEMPLEADOS'].fillna(0)
tabla_edad['TD'] = (tabla_edad['DESEMPLEADOS']/tabla_edad['PEA'])*100

tabla_edad = tabla_edad.reset_index()
tabla_edad['Departamento'] = tabla_edad['DPTO'].map(dpto_nombre)

# Quitamos el grupo de 50 a 60 que el taller no pide
tabla_edad = tabla_edad[tabla_edad['grupo_edad']!='50 a 60']
tabla_edad = tabla_edad[['Departamento','grupo_edad','PEA','DESEMPLEADOS','TD']]
tabla_edad

In [0]:
# Grupo de edad con mayor tasa de desempleo en cada departamento
mayor_edad = tabla_edad.loc[tabla_edad.groupby('Departamento')['TD'].idxmax()]
mayor_edad[['Departamento','grupo_edad','TD']]

In [0]:
# Grupo de edad con menor tasa de desempleo en cada departamento
menor_edad = tabla_edad.loc[tabla_edad.groupby('Departamento')['TD'].idxmin()]
menor_edad[['Departamento','grupo_edad','TD']]

**Respuesta:**

En casi todos los departamentos, los jóvenes de 18 a 24 años son los que más desempleo tienen, y las personas de 60 años en adelante son las que menos, tambien hay algunas excepciones, como Arauca, Boyacá, Chocó, Nariño, Putumayo y Vaupés, donde el grupo con más desempleo es el de 24 a 30 años. En el Quindío pasa lo mismo que en la mayoría: los de 18 a 24 años tienen el desempleo más alto, con 15,6%, y los de 60 en adelante el más bajo, con 3,1%

## 4. ¿Cuáles son los 3 departamentos con menor tasa de desempleo de mujeres entre los 18 y los 25 años?

In [0]:
# Desempleadas entre 18 y 25 años por departamento
desempleadas_18_25 = desempleados[(desempleados['P3271']==2) & (desempleados['P6040']>=18) & (desempleados['P6040']<25) & (desempleados['DSI']==1)].groupby('DPTO')['FEX_C18'].sum()

# Mujeres de 18 a 25 años en la PEA por departamento
pea_mujeres_18_25 = ft[(ft['P3271']==2) & (ft['P6040']>=18) & (ft['P6040']<25) & (ft['FT']==1)].groupby('DPTO')['FEX_C18'].sum()

tabla_mujeres = pd.DataFrame({'PEA': pea_mujeres_18_25, 'DESEMPLEADAS': desempleadas_18_25})
tabla_mujeres['DESEMPLEADAS'] = tabla_mujeres['DESEMPLEADAS'].fillna(0)
tabla_mujeres['TD'] = (tabla_mujeres['DESEMPLEADAS']/tabla_mujeres['PEA'])*100

tabla_mujeres = tabla_mujeres.reset_index()
tabla_mujeres['Departamento'] = tabla_mujeres['DPTO'].map(dpto_nombre)
tabla_mujeres = tabla_mujeres[['Departamento','PEA','DESEMPLEADAS','TD']]

# Ordenamos de menor a mayor para ver los de menor desempleo primero
tabla_mujeres = tabla_mujeres.sort_values(by='TD', ascending=True)
tabla_mujeres.head(3)

**Respuesta:**

Los tres departamentos donde las mujeres de 18 a 25 años tienen menos desempleo son Amazonas y Vaupés, con 0%, y Nariño, con 4,1%. Hay que tener en cuenta que en Amazonas y Vaupés hay muy pocas mujeres de esa edad en la muestra, así que ese 0% no es muy confiable como tal el resultado más sólido es el de Nariño

## 5. Promedio de ingresos laborales (INGLABO) en Risaralda, Caldas, Quindío, Antioquia y Valle del Cauca

In [0]:
# Códigos: Risaralda 66, Caldas 17, Quindio 63, Antioquia 5, Valle del Cauca 76
dptos_interes = [66, 17, 63, 5, 76]

ingresos_dpto = ocupados[ocupados['DPTO'].isin(dptos_interes)].groupby('DPTO')['INGLABO'].mean()

ingresos_dpto = ingresos_dpto.reset_index()
ingresos_dpto['Departamento'] = ingresos_dpto['DPTO'].map(dpto_nombre)
ingresos_dpto = ingresos_dpto[['Departamento','INGLABO']].sort_values(by='INGLABO', ascending=False)
ingresos_dpto

**Respuesta:**

Antioquia es el departamento con mejores ingresos laborales, con un promedio de 2,75 millones de pesos, de ahi le siguen Caldas con 2,43 millones, Quindío con 2,29 millones, Risaralda con 2,26 millones y Valle del Cauca con 2,22 millones, que es el más bajo

## 6. Promedio de ingresos laborales de hombres y mujeres por rango de edad en los mismos departamentos

In [0]:
# Grupo de edad también en ocupados
ocupados['grupo_edad'] = pd.cut(ocupados['P6040'], bins=rangos, labels=nombres_grupos, right=False)

ingresos_dpto_sexo_edad = ocupados[ocupados['DPTO'].isin(dptos_interes) & (ocupados['grupo_edad']!='50 a 60')].groupby(['DPTO','P3271','grupo_edad'], observed=True)['INGLABO'].mean()

ingresos_dpto_sexo_edad = ingresos_dpto_sexo_edad.reset_index()
ingresos_dpto_sexo_edad['Departamento'] = ingresos_dpto_sexo_edad['DPTO'].map(dpto_nombre)
ingresos_dpto_sexo_edad['Sexo'] = ingresos_dpto_sexo_edad['P3271'].map(sexo_nombre)
ingresos_dpto_sexo_edad = ingresos_dpto_sexo_edad[['Departamento','Sexo','grupo_edad','INGLABO']]
ingresos_dpto_sexo_edad

**Respuesta:**

En casi todos los departamentos y edades los hombres ganan más que las mujeres. Solo en el grupo de 24 a 30 años, en Antioquia y Risaralda, las mujeres ganan un poco más, se ve que los mejores ingresos están entre los 30 y los 50 años, y los más bajos en los jóvenes de 18 a 24 años y en las personas de 60 en adelante, sobre todo en las mujeres

## 7. ¿Cuáles son los oficios con mayores promedios salariales a nivel nacional?

In [0]:
# Promedio de ingresos por oficio y cuántas personas hay en cada uno
ingresos_oficio = ocupados.groupby('OFICIO_C8')['INGLABO'].agg(promedio='mean', personas='count')
ingresos_oficio = ingresos_oficio.sort_values(by='promedio', ascending=False)
ingresos_oficio.head(10)

**Respuesta:**
Los oficios que mejor pagan son los códigos 2619, 2612 y 2212, que según el DANE son profesionales del derecho, jueces y médicos especialistas, con promedios de 16,5, 14,2 y 12,3 millones de pesos, y Los dos primeros tienen muy pocas personas en la muestra, así que el más confiable es el de médicos especialistas


## 8. Elegir un código de oficio y concluir si hay diferencias significativas entre los salarios de hombres y mujeres en Caldas, Quindío, Antioquia y Valle del Cauca

In [0]:
# Elegimos el oficio más común en esos cuatro departamentos para tener suficientes datos
dptos_8 = [17, 63, 5, 76]
oficio = ocupados[ocupados['DPTO'].isin(dptos_8)]['OFICIO_C8'].value_counts().index[0]
print(f"el oficio elegido es el código {oficio}")

In [0]:
# Promedio de ingresos de hombres y mujeres para ese oficio en cada departamento
datos_oficio = ocupados[(ocupados['OFICIO_C8']==oficio) & (ocupados['DPTO'].isin(dptos_8))]

promedios_oficio = datos_oficio.groupby(['DPTO','P3271'])['INGLABO'].agg(promedio='mean', personas='count').reset_index()
promedios_oficio['Departamento'] = promedios_oficio['DPTO'].map(dpto_nombre)
promedios_oficio['Sexo'] = promedios_oficio['P3271'].map(sexo_nombre)
promedios_oficio[['Departamento','Sexo','promedio','personas']]

In [0]:
# Prueba para saber si la diferencia entre hombres y mujeres es significativa en cada departamento
for d in dptos_8:
    h = datos_oficio[(datos_oficio['DPTO']==d) & (datos_oficio['P3271']==1)]['INGLABO'].dropna()
    m = datos_oficio[(datos_oficio['DPTO']==d) & (datos_oficio['P3271']==2)]['INGLABO'].dropna()
    prueba = stats.ttest_ind(h, m, equal_var=False)
    print(f"{dpto_nombre[d]}: hombres {h.mean():.0f}, mujeres {m.mean():.0f}, p-valor {prueba.pvalue:.3f}")

**Respuesta:**

miramos el oficio 5223, que corresponde a vendedores de tiendas y almacenes, ya que en Antioquia los hombres ganan mucho más que las mujeres, 2,44 millones contra 1,42 millones, y esa diferencia se nota. En Valle del Cauca los hombres también ganan más, pero la diferencia no es tan segura. En Caldas y Quindío los ingresos son parecidos y las diferencias pueden ser casualidad, incluso en Quindío las mujeres ganan un poco más, entonces solo en Antioquia se puede decir con seguridad que hay una diferencia entre hombres y mujeres

## 9. En el Quindío ¿cuál es el oficio con mayor y menor promedio salarial en hombres y mujeres?

In [0]:
# Promedio de ingresos por oficio y sexo en el Quindío
quindio = ocupados[ocupados['DPTO']==63].groupby(['P3271','OFICIO_C8'])['INGLABO'].agg(promedio='mean', personas='count').reset_index()
quindio['Sexo'] = quindio['P3271'].map(sexo_nombre)
quindio = quindio[['Sexo','OFICIO_C8','promedio','personas']]

# Mayor y menor promedio para cada sexo
mayor_q = quindio.loc[quindio.groupby('Sexo')['promedio'].idxmax()]
menor_q = quindio.loc[quindio.groupby('Sexo')['promedio'].idxmin()]

print("Oficio con mayor promedio")
print(mayor_q)
print("Oficio con menor promedio")
print(menor_q)

**Respuesta:**

En el Quindío, los hombres ganan más en el oficio 2145, ingenieros químicos, con 23 millones de pesos, y menos en el oficio 9611, recolectores de basura, con 100 mil pesos, y ya ahi las mujeres ganan más en el oficio 2512, desarrolladores de software, con 10 millones, y menos en el oficio 3142, técnicos agropecuarios, con ingresos de cero y ya estos datos salen de muy pocas personas por oficio, así que sirven de ejemplo pero no representan a todo el Quindío